# Hafta 8 — Topluluk Yöntemleri (Ensemble Methods)

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy xgboost lightgbm

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Zor bir soruda tek bir kişinin tahmini yanıltıcı olabilir. Kalabalık bir grubun ortak tahmini ise çoğu zaman gerçeğe çok daha yakındır. Topluluk yöntemleri bu fikirden yararlanır:

- **Random Forest (Rastgele Orman):** Yüzlerce karar ağacı ayrı ayrı tahmin yapar ve en çok oyu alan cevap seçilir.
- **Boosting:** Modeller sırayla eğitilir ve her yeni model, bir öncekinin hata yaptığı örneklere odaklanır. Böylece takım zayıf noktalarını adım adım kapatır.

---

## Bu Haftanın Hedefleri

- Birden çok modeli birleştirmenin neden işe yaradığını görmek
- Bagging ve Random Forest'ın çalışma mantığını anlamak
- AdaBoost ve Gradient Boosting'i elle bir tur çalıştırmak
- Tablo verisinde modelleri karşılaştırmak

---

## 1. Neden Topluluk?

Her biri bağımsız olarak %70 doğrulukla çalışan 5 model düşünelim. Çoğunluk oylamasında en az 3'ünün doğru olma olasılığı:

```
P(≥ 3 doğru) = 10·0.7³·0.3² + 5·0.7⁴·0.3 + 0.7⁵ = 0.837
```

Tek model %70, beş modelin oylaması **%84**. Şart şudur: Modellerin hataları birbirinden **farklı** olmalıdır. Aynı hatayı yapan beş kopya modelin oylaması hiçbir şey kazandırmaz.

| | Bagging | Boosting |
|---|---|---|
| Eğitim | Paralel, birbirinden bağımsız | Sıralı, her model öncekinin hatasına odaklanır |
| Temel model | Derin ağaçlar | Sığ ağaçlar |
| Azalttığı sorun | Varyans | Yanlılık |

---

## 2. Bagging ve Random Forest

### Bootstrap örnekleme

n örnekten **yerine koyarak** n örnek çekilir. Bazı örnekler birkaç kez seçilir, bazıları hiç seçilmez:

```
Orijinal:     [A, B, C, D, E]
Bootstrap 1:  [A, A, C, D, E]     B dışarıda
Bootstrap 2:  [B, C, C, C, E]     A ve D dışarıda
```

Her bootstrap örneği verinin yaklaşık **%63**'ünü içerir. Dışarıda kalan örneklere **OOB** denir ve modeli ücretsiz doğrulamak için kullanılır.

### Algoritma

```
Bagging:       Her ağaç farklı bir bootstrap örneğiyle eğitilir → tahminler oylanır.
Random Forest: Bagging + her bölmede özniteliklerin yalnızca rastgele bir alt kümesine bakılır.
```

Random Forest'taki ek rastgelelik ağaçları birbirinden farklılaştırır. Çok güçlü bir öznitelik bile her bölmede aday olmadığı için ağaçların hepsi aynı kökle başlamaz.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

class BenimRandomForest:
    def __init__(self, n_agac=100, seed=0):
        self.n_agac, self.seed = n_agac, seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        self.agaclar = []
        for i in range(self.n_agac):
            idx = rng.integers(0, len(X), len(X))                      # bootstrap
            agac = DecisionTreeClassifier(max_features="sqrt", random_state=i)
            self.agaclar.append(agac.fit(X[idx], y[idx]))
        return self

    def predict(self, X):
        oylar = np.mean([a.predict_proba(X) for a in self.agaclar], axis=0)
        return oylar.argmax(axis=1)

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print("Tek ağaç     :", DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr).score(X_te, y_te))
print("Random Forest:", (BenimRandomForest().fit(X_tr, y_tr).predict(X_te) == y_te).mean())

Ağaç sayısını artırmak Random Forest'ta aşırı öğrenmeye yol açmaz; sadece hesaplama süresini uzatır.

---

## 3. AdaBoost

### Çalışma mantığı

1. Bütün örneklere eşit ağırlık ver.
2. Ağırlıklı veride basit bir model (tek soruluk ağaç) eğit.
3. Modelin ağırlıklı hatasını (ε) hesapla. Modelin söz hakkı: **α = ½·ln((1 − ε) / ε)**
4. Yanlış bilinen örneklerin ağırlığını **artır**, doğru bilinenlerinkini **azalt**.
5. Tekrarla. Son karar, modellerin α ile ağırlıklandırılmış oylamasıdır.

### Bir tur elle

10 örnek, her birinin ağırlığı 0.1. İlk model 3 örneği yanlış biliyor:

```
ε = 3 × 0.1 = 0.3
α = ½ · ln(0.7 / 0.3) = 0.42
Yanlış bilinen her örnek:  0.1 → 0.167  (ağırlık arttı)
Doğru bilinen her örnek:   0.1 → 0.071  (ağırlık azaldı)
```

Bir sonraki model, artık daha ağır olan bu 3 zor örneğe odaklanmak zorunda kalır.

---

## 4. Gradient Boosting

**Fikir:** Her yeni ağaç, önceki modelin **kalan hatasını** (artığını) tahmin etmeye çalışır.

```
F₀ = ortalama
Her adımda:  artık = gerçek − mevcut tahmin
             artıklara küçük bir ağaç uydur
             yeni tahmin = mevcut tahmin + η × ağacın tahmini      (η: öğrenme oranı)
```

### Elle örnek (η = 0.5)

| Ev | Metrekare | Fiyat | F₀ | Artık | F₁ | Yeni artık |
|---|---|---|---|---|---|---|
| A | 70 | 2.0 | 3.0 | −1.0 | 2.6 | −0.6 |
| B | 90 | 2.4 | 3.0 | −0.6 | 2.6 | −0.2 |
| C | 120 | 3.6 | 3.0 | +0.6 | 3.4 | +0.2 |
| D | 150 | 4.0 | 3.0 | +1.0 | 3.4 | +0.6 |

İlk ağaç "metrekare ≤ 105" sorusunu sordu ve artıklar için −0.8 (A, B) ile +0.8 (C, D) tahmin etti. Tahminler bu değerlerin yarısı kadar düzeltildi. Hata (MSE) 0.68'den 0.20'ye düştü. Her yeni ağaç kalan hatayı biraz daha küçültür.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

def gradient_boosting(X, y, n_agac=100, eta=0.1, derinlik=2):
    F = np.full(len(y), y.mean())
    agaclar = []
    for _ in range(n_agac):
        agac = DecisionTreeRegressor(max_depth=derinlik).fit(X, y - F)   # artıklara uydur
        F += eta * agac.predict(X)
        agaclar.append(agac)
    return y.mean(), agaclar

def gb_tahmin(model, X, eta=0.1):
    baslangic, agaclar = model
    return baslangic + eta * sum(a.predict(X) for a in agaclar)

Xe = np.array([[70], [90], [120], [150]]); ye = np.array([2.0, 2.4, 3.6, 4.0])
model = gradient_boosting(Xe, ye, n_agac=2, eta=0.5, derinlik=1)
print(gb_tahmin(model, Xe, eta=0.5))       # [2.4 2.4 3.6 3.6]

Random Forest'tan farklı olarak ağaç sayısı çok artarsa Gradient Boosting aşırı öğrenebilir. Bu yüzden küçük bir öğrenme oranı ve **erken durdurma** kullanılır.

**XGBoost, LightGBM ve CatBoost**, Gradient Boosting'in hızlandırılmış ve düzenlileştirilmiş sürümleridir. Tablo verisinde genellikle en yüksek başarıyı verirler.

---

## 5. Laboratuvar: Model Karşılaştırması

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier

modeller = {
    "Lojistik Regresyon": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "Karar Ağacı": DecisionTreeClassifier(random_state=0),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=0),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=0),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=0),
}
for ad, m in modeller.items():
    skor = cross_val_score(m, X, y, cv=5, scoring="roc_auc")
    print(f"{ad:20s} ROC-AUC = {skor.mean():.4f} ± {skor.std():.4f}")

**Görevler:**
1. Tek karar ağacı ile Random Forest arasındaki farkı yanlılık-varyans açısından yorumlayın.
2. En iyi Random Forest modelinin öznitelik önemlerini (`feature_importances_`) çubuk grafikle gösterin.

---

## 6. Özet

- Topluluklar, farklı modellerin hatalarının birbirini götürmesinden güç alır.
- **Random Forest:** bootstrap + rastgele öznitelikler + oylama. Varyansı azaltır.
- **AdaBoost:** Yanlış bilinen örneklerin ağırlığını artırır.
- **Gradient Boosting:** Her ağaç önceki modelin artıklarını düzeltir.

## Kendinizi Sınayın

1. AdaBoost'ta ε = 0.2 ise α kaçtır?
2. Random Forest'ta neden derin, boosting'de neden sığ ağaçlar kullanılır?
3. Gradient Boosting'de öğrenme oranını küçültürseniz ağaç sayısını nasıl değiştirmelisiniz?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.